# 08 Monitoring Basics

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Monitor a Databricks platform at three levels: <b>queries</b> (query history and profiles), <b>jobs</b> (run history, durations, failures) and <b>data</b> (table history, freshness, volume). Use <b>system tables</b> to analyse usage and cost with SQL, and build simple checks that alert when something looks wrong.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Pipelines rarely fail loudly: more often they get slower, cost more, or quietly stop updating a table. Monitoring catches that before users do. The 2026 exam has a dedicated "Troubleshooting, Monitoring and Optimization" domain (about 10%), and "how do you monitor your pipelines?" is a common interview question.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Where monitoring information lives</b><br>
| Question | Where to look |<br>|---|---|<br>| Which query was slow, and why? | Query History → query profile (lesson 02_14) |<br>| Did my job run, how long did it take, what failed? | Jobs & Pipelines → job → Runs (matrix view, run details, logs) |<br>| When was this table last updated, by what? | <code>DESCRIBE HISTORY</code>, Catalog Explorer → History, lineage |<br>| How much compute did we use, and on what? | <code>system.billing.usage</code> (+ <code>system.billing.list_prices</code>) |<br>| Which queries run, by whom, how long? | <code>system.query.history</code> |<br>| Job runs across the workspace | <code>system.lakeflow.job_run_timeline</code>, <code>system.lakeflow.jobs</code> |<br>| Who accessed what? | <code>system.access.audit</code> |<br>| Data quality over time | Lakehouse monitoring (data profiling), expectations metrics, your own checks |
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
System tables are in the <code>system</code> catalog. Which ones you can read depends on what has been enabled and granted in your workspace. Every cell below handles the case where a table isn't accessible.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A finance team noticed their monthly Databricks bill had doubled. A query on <code>system.billing.usage</code>, grouped by job and SKU, showed that one job had started running every 5 minutes instead of every hour after a schedule edit. A second query on <code>system.lakeflow.job_run_timeline</code> confirmed the change date. They fixed the schedule and added a weekly cost report to catch such changes early.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a small table and changes it a few times, so there's history to monitor.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table with several versions.
</div>

In [0]:
DB = "workspace.platform"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

TABLE = f"{DB}.monitored_orders"
spark.sql(f"CREATE OR REPLACE TABLE {TABLE} (order_id INT, amount DOUBLE, loaded_at TIMESTAMP)")
for batch in range(3):
    spark.sql(f"""INSERT INTO {TABLE}
                  SELECT id + {batch * 100}, rand() * 100, current_timestamp() FROM range({50 + batch * 25})""")
spark.sql(f"DELETE FROM {TABLE} WHERE amount < 5")
print("versions:", spark.sql(f"DESCRIBE HISTORY {TABLE}").count())

## 1. Data monitoring: freshness and volume from the table itself

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>DESCRIBE HISTORY</code> to report when the table last changed, how each write changed it, and checks freshness and volume against thresholds.<br><br>
<b>Why it matters</b><br>The most common silent failure is a table that simply <b>stops being updated</b>. A freshness check ("was it updated in the last N hours?") and a volume check ("did the last load write a normal number of rows?") catch most of them, using only Delta's own history.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The last write time and operation, rows written per load, then <code>freshness: OK</code> and a volume result for the latest load compared with the average of the previous ones.
</div>

In [0]:
history = spark.sql(f"DESCRIBE HISTORY {TABLE}")
history.select("version", "timestamp", "operation", "operationMetrics.numOutputRows", "operationMetrics.numDeletedRows") \
       .orderBy("version").show(truncate=False)

def hours_since_last_change(history_df):
    """Age of the latest commit, computed in Spark (no time zone conversions in Python)."""
    return history_df.agg(((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.max("timestamp"))) / 3600)
                          .alias("h")).first()["h"]

last = history.orderBy(F.desc("version")).first()
age_hours = hours_since_last_change(history)
print(f"last change: version {last['version']} ({last['operation']}) {age_hours:.2f} hours ago")
print("freshness:", "OK" if age_hours <= 24 else "STALE")

loads = [int(r["rows"]) for r in history.filter("operation = 'WRITE'").orderBy("version")
         .select(F.col("operationMetrics.numOutputRows").alias("rows")).collect()]
latest, previous = loads[-1], loads[:-1]
avg_prev = sum(previous) / len(previous)
print(f"volume: latest load {latest} rows vs previous average {avg_prev:.0f}",
      "-> OK" if 0.5 * avg_prev <= latest <= 2 * avg_prev else "-> ANOMALY")

## 2. Query monitoring with system tables

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries <code>system.query.history</code> for your recent statements: duration, rows and bytes read, and status.<br><br>
<b>Why it matters</b><br>System tables turn monitoring into SQL: you can rank the slowest queries, find failing ones, and build dashboards and alerts on them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your latest queries with duration and status, or a message that the table isn't accessible in this workspace.
</div>

In [0]:
def try_sql(sql, n=10):
    try:
        display(spark.sql(sql).limit(n))
    except Exception as e:
        print("not accessible here:", type(e).__name__, str(e).splitlines()[0][:120])

try_sql("""
    SELECT start_time, statement_type, execution_status, total_duration_ms,
           read_rows, read_bytes, left(statement_text, 80) AS statement
    FROM system.query.history
    WHERE executed_by = current_user() AND start_time > current_timestamp() - INTERVAL 1 DAY
    ORDER BY total_duration_ms DESC
""")

## 3. Cost monitoring with system tables

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Summarizes DBU usage by day and product from <code>system.billing.usage</code>, and estimates cost with list prices.<br><br>
<b>Why it matters</b><br>Cost is a metric like any other: knowing which jobs, warehouses and users consume DBUs lets you spot regressions (a schedule change, a runaway query) and prioritize optimization. Lesson 05_07 goes deeper.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
DBUs per day and billing product for the last 30 days, and an estimated list-price cost, or a message if billing tables aren't accessible.
</div>

In [0]:
try_sql("""
    SELECT usage_date, billing_origin_product, round(sum(usage_quantity), 2) AS dbus
    FROM system.billing.usage
    WHERE usage_date >= current_date() - INTERVAL 30 DAYS
    GROUP BY ALL
    ORDER BY usage_date DESC, dbus DESC
""")

try_sql("""
    SELECT u.billing_origin_product,
           round(sum(u.usage_quantity * p.pricing.default), 2) AS est_list_cost
    FROM system.billing.usage u
    JOIN system.billing.list_prices p
      ON u.sku_name = p.sku_name
     AND u.usage_start_time >= p.price_start_time
     AND (p.price_end_time IS NULL OR u.usage_start_time < p.price_end_time)
    WHERE u.usage_date >= current_date() - INTERVAL 30 DAYS
    GROUP BY 1 ORDER BY 2 DESC
""")

## 4. Job monitoring

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists recent job runs with the SDK (state, duration), and the equivalent system table query.<br><br>
<b>Why it matters</b><br>For each production job you want to know: did it run, did it succeed, and how long did it take compared with usual. Duration creeping up is an early warning of data growth or skew.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your recent job runs (empty if you haven't created jobs yet, see <code>07_workflows</code>), and run results from <code>system.lakeflow.job_run_timeline</code> if accessible.
</div>

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
try:
    runs = list(w.jobs.list_runs(limit=10))
    for r in runs:
        duration = (r.end_time - r.start_time) / 1000 if r.end_time and r.start_time else None
        print(f"{r.run_name:<35} {r.state.result_state if r.state else ''}  {duration}s")
    if not runs:
        print("no job runs yet (create one in 07_workflows)")
except Exception as e:
    print("job runs not listable:", type(e).__name__)

try_sql("""
    SELECT job_id, run_id, result_state, period_start_time, period_end_time,
           timestampdiff(SECOND, period_start_time, period_end_time) AS seconds
    FROM system.lakeflow.job_run_timeline
    WHERE period_start_time > current_timestamp() - INTERVAL 7 DAYS
    ORDER BY period_start_time DESC
""")

## 5. A reusable monitoring check with an alert signal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Wraps freshness, volume and failure checks into one function that returns a result table and raises an error when a check fails, so a scheduled job (or a SQL alert) can notify someone.<br><br>
<b>Why it matters</b><br>Monitoring only helps if someone is told. The simplest robust pattern: a small scheduled job runs checks and <b>fails</b> when something is wrong, and the job's failure notification sends the email or Slack message. Databricks SQL <b>alerts</b> do the same for a SQL query with a threshold.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A small results table with each check's status. With the thresholds used here, all checks pass.
</div>

In [0]:
def table_health(table, max_age_hours=24, min_rows=1):
    h = spark.sql(f"DESCRIBE HISTORY {table}")
    last = h.orderBy(F.desc("version")).first()
    age = hours_since_last_change(h)
    rows = spark.table(table).count()
    checks = [
        ("freshness", f"{age:.2f}h since last change", age <= max_age_hours),
        ("not empty", f"{rows} rows", rows >= min_rows),
        ("last operation", last["operation"], last["operation"] not in ("RESTORE",)),
    ]
    return spark.createDataFrame(checks, "check STRING, observed STRING, ok BOOLEAN")

results = table_health(TABLE)
results.show(truncate=False)
failed = [r["check"] for r in results.collect() if not r["ok"]]
if failed:
    raise RuntimeError(f"health checks failed for {TABLE}: {failed}")   # a job failure triggers its notifications
print("all checks passed")

## 6. Lakehouse monitoring (data profiling)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Describes Databricks' built-in table monitoring, which profiles a table over time and detects drift.<br><br>
<b>Why it matters</b><br>Instead of writing every check yourself, a <b>monitor</b> on a Unity Catalog table computes profile metrics (null rates, distributions, counts) on a schedule, stores them in metric tables, and generates a dashboard. Snapshot, time-series and inference (ML) profiles are supported. Create one from the table's <b>Quality</b> tab in Catalog Explorer, or with the SDK (<code>w.quality_monitors.create</code>).
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Availability of table monitors depends on your workspace. If you can't create one, the custom checks in section 5 give you the essentials.
</div>

## Under the hood

```
Everything you run ──▶ logs and metrics ──▶ system tables (system.*), Delta history, job run records
                                               │
                    SQL queries / dashboards / alerts / health-check jobs
                                               │
                         notifications (email, Slack, webhooks) to the owners
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> `DESCRIBE HISTORY` reads only the Delta log, so freshness checks are cheap.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> system table queries run on compute and cost a little. Keep them filtered by date.

In [0]:
spark.sql(f"DESCRIBE DETAIL {TABLE}").select("numFiles", "sizeInBytes", "lastModified").show(truncate=False)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: a table stopped updating and nobody noticed for a week</b><br>
No freshness check. Add one per critical table, with an alert.<br><br>
**⛔ Problem: <code>TABLE_OR_VIEW_NOT_FOUND</code> / permission denied on <code>system.*</code>**<br>The system schema isn't enabled or you weren't granted access. Ask an admin, and use the UI pages meanwhile.<br><br>
<b>⛔ Problem: alerts on every small fluctuation</b><br>Thresholds are too tight. Compare with a rolling average and alert on large deviations.<br><br>
<b>⛔ Problem: monitoring queries are expensive</b><br>They scan whole system tables. Always filter by date.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you monitor data pipelines on Databricks?</b><br>
At three levels. Jobs: run history, durations and failure notifications on every job. Queries: query history and profiles for slow statements. Data: freshness, volume and quality checks on key tables, using Delta history, expectations, table monitors and custom checks that fail a job to trigger alerts. System tables give a SQL view across all of it, including cost.<br><br>

<b>🎤 2. What are system tables?</b><br>
Databricks-managed tables in the <code>system</code> catalog with operational data: billing usage and prices, query history, job runs, audit logs, lineage and more. They let you analyse cost, performance and access with SQL and build dashboards and alerts.<br><br>

<b>🎤 3. How would you find which job causes a cost increase?</b><br>
Query <code>system.billing.usage</code> grouped by day and by usage metadata such as the job ID, compare periods, and join <code>system.billing.list_prices</code> for cost. Then check that job's run timeline for changes in frequency or duration.<br><br>

<b>🎤 4. How do you detect that a table is no longer being updated?</b><br>
With a freshness check: compare the latest commit timestamp from <code>DESCRIBE HISTORY</code> (or the latest event time in the data) with the expected schedule, and alert when it's older than the threshold.<br><br>

<b>🎤 5. How do you send alerts?</b><br>
Job and task notifications on failure, duration thresholds or success, Databricks SQL alerts on query results, and health-check jobs that fail when a check fails, routed to email, Slack or webhooks.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A team wants to analyse DBU consumption per job over the last month using SQL. Which source should they query?</b><br>
A. <code>DESCRIBE HISTORY</code><br>
B. <code>system.billing.usage</code><br>
C. The Spark UI<br>
D. <code>information_schema.tables</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which command shows when a Delta table was last written and by which operation?</b><br>
A. <code>DESCRIBE DETAIL</code><br>
B. <code>DESCRIBE HISTORY</code><br>
C. <code>SHOW TABLES</code><br>
D. <code>ANALYZE TABLE</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>DESCRIBE DETAIL</code> shows <code>lastModified</code>, but not the operations.</details><br><br>

<b>Q3. How can a data engineer be notified automatically when a scheduled job fails?</b><br>
A. Check the Jobs page every morning<br>
B. Configure failure notifications (email, Slack or webhook) on the job<br>
C. Enable change data feed<br>
D. Run <code>VACUUM</code> after each run<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a function <code>freshness_report(tables, max_age_hours)</code> that returns one row per table with its last change time and status</li><li>Write a SQL query on <code>system.query.history</code> that returns your 5 slowest queries of the last 7 days</li><li>Write a SQL query on <code>system.billing.usage</code> that returns DBUs per day for the last 14 days, and describe how you'd turn it into a weekly email (dashboard schedule or alert)</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Freshness for several tables
def freshness_report(tables, max_age_hours=24):
    rows = []
    for t in tables:
        try:
            h = spark.sql(f"DESCRIBE HISTORY {t}")
            last = h.agg(F.max("timestamp").cast("string")).first()[0]
            age = hours_since_last_change(h)
            rows.append((t, last, round(age, 2), "OK" if age <= max_age_hours else "STALE"))
        except Exception as e:
            rows.append((t, None, None, f"ERROR: {type(e).__name__}"))
    return spark.createDataFrame(rows, "table STRING, last_change STRING, age_hours DOUBLE, status STRING")

spark.sql(f"CREATE OR REPLACE TABLE {TABLE} AS SELECT 1 AS x")
freshness_report([TABLE, f"{DB}.does_not_exist"]).show(truncate=False)
spark.sql(f"DROP TABLE {TABLE}")

# 2. Slowest queries (run on Databricks):
print("""SELECT left(statement_text, 100) AS statement, total_duration_ms, start_time
FROM system.query.history
WHERE executed_by = current_user() AND start_time > current_timestamp() - INTERVAL 7 DAYS
ORDER BY total_duration_ms DESC LIMIT 5""")

# 3. DBUs per day; schedule it as a dashboard email snapshot, or create a SQL alert on a threshold.
print("""SELECT usage_date, round(sum(usage_quantity), 2) AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - INTERVAL 14 DAYS
GROUP BY usage_date ORDER BY usage_date""")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Monitor jobs (runs, durations, failures), queries (history, profiles) and data (freshness, volume, quality)</li><li><code>DESCRIBE HISTORY</code> gives cheap freshness and volume signals for Delta tables</li><li>System tables (<code>system.billing.*</code>, <code>system.query.history</code>, <code>system.lakeflow.*</code>, <code>system.access.audit</code>) make monitoring SQL</li><li>Alert through job notifications, SQL alerts, or health-check jobs that fail</li><li>Table monitors (data profiling) track quality and drift over time</li></ul>
</div>

| Question | Source |
|---|---|
| Slow queries | `system.query.history`, Query History UI |
| Cost | `system.billing.usage` + `system.billing.list_prices` |
| Job runs | `system.lakeflow.job_run_timeline`, Jobs UI, `w.jobs.list_runs()` |
| Table freshness | `DESCRIBE HISTORY t` |
| Access audit | `system.access.audit` |

## Git commit

```
git add 04_databricks/08_monitoring_basics.ipynb
git commit -m "add 04_08 monitoring basics notebook"
```